# [재학습 · 2026-10-06] β=30 강건성 검증 (B 묶음)

재학습 후 검증 샤프가 가장 높은 β=30을 새 배포 후보로 두고, 원본 노트북 셋을 β만 바꿔 그대로 다시 실행한다(GitHub 커밋 `0e86a41`의 코드).
1. 추가 시드 15개 + 앙상블 + DSR (`crypto_beta_robustness_colab`)
2. 클린 재학습: 1~9월만 학습, 10~12월에서 α 선택 (`crypto_clean_valid_colab`)
3. top-K 스윕 K∈{5,7,15} + IC (`crypto_hrl_earnhft_colab` §17-C)

- **`crypto_cascade_retrain_rt1006`의 v4 블록이 끝난 뒤에 실행해야 한다**(재학습한 β=30 모델 필요 — 이미 끝남).
- **`런타임 > 모두 실행`만 누르면 된다.** 끊기면 다시 `모두 실행`. PPO 19개, 약 5~6시간.
- 드라이브 연결 창에서 **ksbdaniel7@gmail.com** 계정을 고른다.
- 결과 CSV는 원본과 같은 이름으로 `data/crypto/rt1006/`에 저장된다(라벨의 'β=-30' 문구는 원본 그대로라 β=30으로 읽으면 된다).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q stable-baselines3[extra] gymnasium h5py pyarrow


In [ ]:
# 실행기 — GitHub의 원본 노트북(커밋 0e86a41 고정)을 불러와 셀을 순서대로 실행한다.
# (1) 설정 셀 직후 모델·결과 경로를 새 폴더(rt1006)로 바꿔 기존 결과를 덮어쓰지 않는다.
# (2) 모든 .learn() 호출 결과를 캐시에 저장 — 끊겨도 `모두 실행`을 다시 누르면 학습된 부분은 건너뛴다.
import json, urllib.request
from pathlib import Path
COMMIT = '0e86a41a701841d9a17398a8cebf240cd0c40218'
RAW = f'https://raw.githubusercontent.com/winbean-kwon/ToBeRich/{COMMIT}/'
TAG = 'rt1006'
_ip = get_ipython()

def fetch_cells(path):
    nb = json.loads(urllib.request.urlopen(RAW + path).read().decode('utf-8'))
    return {i: (c['source'] if isinstance(c['source'], str) else ''.join(c['source']))
            for i, c in enumerate(nb['cells']) if c['cell_type'] == 'code'}

PATCH = r"""
MODEL_DIR = DRIVE_ROOT / 'models' / TAG
CRYPTO_DIR = DATA_DIR / 'crypto' / TAG
MODEL_DIR.mkdir(parents=True, exist_ok=True); CRYPTO_DIR.mkdir(parents=True, exist_ok=True)
POOL_DIR = MODEL_DIR / 'crypto_hrl_pool'
ORIG_PPO_PATH = MODEL_DIR / 'crypto_ppo_portfolio.zip'
for _n in ('EIIE_MODEL_DIR', 'LSRE_CAAN_MODEL_DIR'):
    if _n in globals():
        globals()[_n] = MODEL_DIR / Path(globals()[_n]).name
        globals()[_n].mkdir(parents=True, exist_ok=True)
CACHE = MODEL_DIR / '_learn_cache'; CACHE.mkdir(exist_ok=True)
import stable_baselines3 as _sb3
_BASE_PPO, _BASE_DQN = _sb3.PPO, _sb3.DQN
def _cached(base):
    class _C(base):
        def learn(self, total_timesteps, *a, **k):
            _CTR[0] += 1
            p = CACHE / f'{_NB[0]}_c{_CUR[0]}_{_CTR[0]}_{base.__name__}.zip'
            if p.exists():
                self.set_parameters(str(p), exact_match=True, device=self.device)
                self.num_timesteps += total_timesteps
                print(f'[캐시] {p.name} 로드 — 학습 생략')
                return self
            out = super().learn(total_timesteps, *a, **k)
            self.save(str(p))
            return out
    _C.__name__ = _C.__qualname__ = base.__name__
    return _C
PPO, DQN = _cached(_BASE_PPO), _cached(_BASE_DQN)
print(f'[경로] 모델 → {MODEL_DIR} | 결과 CSV → {CRYPTO_DIR}')
"""

_NB, _CUR, _CTR = [''], [0], [0]

def run_cells(path, ids, patch_after, extra=None, subs=None):
    """extra: 경로 변경 직후 실행할 코드(예: β 변경). subs: {셀번호: [(원문, 대체), ...]} 문자열 치환."""
    cells = fetch_cells(path)
    _NB[0] = Path(path).stem
    for i in ids:
        _CUR[0], _CTR[0] = i, 0
        src = cells[i]
        for old, new in (subs or {}).get(i, []):
            assert old in src, f'셀 {i}에 치환 대상 없음: {old!r}'
            src = src.replace(old, new)
        print(f'\n########## {Path(path).name} 셀 {i} ##########')
        r = _ip.run_cell(src)
        if not r.success:
            raise RuntimeError(f'{path} 셀 {i}에서 오류 — 위 오류 메시지를 확인하세요')
        if i == patch_after:
            for code in (PATCH, extra):
                if code and not _ip.run_cell(code).success:
                    raise RuntimeError('경로/설정 변경 실패')
print('실행기 준비 완료')


In [ ]:
# 1. 추가 시드 15개 + 앙상블 + DSR (β=30)
run_cells('crypto/notebooks/crypto_beta_robustness_colab.ipynb', [3, 5, 6, 8, 10, 12, 13, 15, 17, 19, 20, 22, 24, 26, 29, 30, 31, 32, 35, 36, 39, 40, 41, 43, 45], patch_after=3, extra="BETA = 30   # 재학습 후 검증 샤프 최고 β\nPOOL_DIR_ROBUST = MODEL_DIR / 'crypto_beta_robustness'\nDEPLOYED_AGENT_PATH = MODEL_DIR / 'crypto_hrl_pool_v4' / f'agent_beta_{BETA}.zip'\nassert DEPLOYED_AGENT_PATH.exists(), f'재학습한 v4 β={BETA} 모델 없음: {DEPLOYED_AGENT_PATH}'\nprint(f'[β 변경] BETA={BETA} | 기준 모델: {DEPLOYED_AGENT_PATH}')\n")


In [ ]:
# 2. 클린 재학습 (β=30, seed=42, 2025년 1~9월만 학습)
run_cells('crypto/notebooks/crypto_clean_valid_colab.ipynb', [3, 5, 6, 8, 10, 12, 13, 15, 17, 19, 20, 22, 24], patch_after=3, extra="BETA = 30\nDEPLOYED_AGENT_PATH = MODEL_DIR / 'crypto_hrl_pool_v4' / f'agent_beta_{BETA}.zip'\nCLEAN_MODEL_DIR = MODEL_DIR / 'crypto_clean_valid'\nassert DEPLOYED_AGENT_PATH.exists(), f'재학습한 v4 β={BETA} 모델 없음: {DEPLOYED_AGENT_PATH}'\nprint(f'[β 변경] BETA={BETA} | 기준 모델: {DEPLOYED_AGENT_PATH}')\n")


In [ ]:
# 3. top-K 스윕 (β=30)
MAIN_PATH = 'crypto/notebooks/crypto_hrl_earnhft_colab.ipynb'
run_cells(MAIN_PATH, [2, 4, 6, 8, 9, 11, 12, 44, 46, 52, 72], patch_after=2, extra="# §17-C가 쓰는 smooth_perf_row(원본 §9 셀 25에 정의)를 그 셀에서 함수 부분만 가져와 정의\n_s25 = fetch_cells(MAIN_PATH)[25]\nexec(_s25[_s25.index('def smooth_perf_row'):_s25.index('smooth_results, smooth_curves = {}')], globals())\nprint('smooth_perf_row 정의 완료')\n")
run_cells(MAIN_PATH, [90, 91], patch_after=None, subs={90: [('pool_models_v4[-30]', 'pool_models_v4[30]'), ('beta=-30,', 'beta=30,'), ("f'agent_beta_-30_k{k}.zip'", "f'agent_beta_30_k{k}.zip'")], 91: [('ALPHA_DIAG = BEST_ALPHA_V4_MATCHED', 'ALPHA_DIAG = 0.01  # 재학습 결과 valid·test 모두 α=0.01 선택')]})
print('\n전부 완료')
